In [ ]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from config import (
    PROJECT_ROOT,
    DATA_DIR,
    INTERIM_DIR,
    PROCESSED_DIR,
    WEATHER_DIR,
    SOIL_DIR,
    RAW_DIR,
    OUTPUT_DIR,
    PROCESSED_DATASET,
    WEATHER_FEATHER,
    MERGED_DATA_DIR,
    interim_csb_path,
)


# mps project - CSB data transform

## 1. Setup and Load Data

In [ ]:
# install the following packages if needed
#%pip install geopandas pandas matplotlib fiona shapely pyproj pyarrow fastparquet

In [ ]:
import os
import geopandas as gpd
import time

# --- Configuration ---
# Set the years for the data you want to process
# Adjust these years to switch between datasets (e.g., 2009-2016 or 2017-2024)
START_YEAR = 2017
END_YEAR = START_YEAR + 7

# Base paths
RAW_DATA_DIR = str(RAW_DIR)
OUTPUT_BASE_DIR = str(DATA_DIR)

# Construct folder and file names based on naming convention
# Folder pattern example: NationalCSB_2009-2016_rev23
# GDB pattern example: CSB0916.gdb (uses last 2 digits of the year)

folder_name = f"NationalCSB_{START_YEAR}-{END_YEAR}_rev23"
gdb_name = f"CSB{str(START_YEAR)[-2:]}{str(END_YEAR)[-2:]}.gdb"

input_path = os.path.join(RAW_DATA_DIR, folder_name, gdb_name)

print(f"Configured to read from: {input_path}")

### Loading the data (This might take 30-60 minutes for large datasets)

In [ ]:
print(f"Starting data load from: {input_path}")
start_time = time.time()

try:
    csb_data = gpd.read_file(input_path)
    print("Data loaded successfully!")
except Exception as e:
    print(f"Error loading data: {e}")
    print(f"Please check if the directory exists: {input_path}")

end_time = time.time()
print(f"Time taken: {(end_time - start_time)/60:.2f} minutes")

## 2. Check if the Data is loaded properly

In [ ]:
if 'csb_data' in locals():
    display(csb_data.head())

In [ ]:
if 'csb_data' in locals():
    display(csb_data.describe())

In [ ]:
if 'csb_data' in locals():
    if 'STATEFIPS' in csb_data.columns:
        unique_states = csb_data['STATEFIPS'].unique()
        print(f"Unique States: {unique_states}")
    else:
        print("STATEFIPS column not found.")

## 3. Store the data into easy to process format

In [ ]:
if 'csb_data' in locals():
    # Create output directory
    output_folder_name = f"csb_{START_YEAR}{END_YEAR}"
    output_dir = os.path.join(OUTPUT_BASE_DIR, output_folder_name)
    
    os.makedirs(output_dir, exist_ok=True)
    print(f"Output directory created/verified: {output_dir}")
    
    # Define filenames
    # Using generic names or including years for clarity
    parquet_path = os.path.join(output_dir, f"ny_csb.parquet_{START_YEAR}{END_YEAR}")
    feather_path = os.path.join(output_dir, f"ny_csb.feather_{START_YEAR}{END_YEAR}")
    csv_path = os.path.join(output_dir, f"ny_csb.csv_{START_YEAR}{END_YEAR}")
    
    print("Preparing data for saving...")
    
    # Filter for New York (STATEFIPS == '36')
    # Note: If you want to save the ENTIRE dataset instead of just NY, 
    # change the line below to: df_to_save = csb_data.copy()
    df_to_save = csb_data[csb_data['STATEFIPS'] == '36'].copy()
    
    print(f"Saving {len(df_to_save)} records to {output_dir}...")

    # 1. Parquet (compressed, good for big data)
    df_to_save.to_parquet(parquet_path, engine='pyarrow')
    print(f"Saved Parquet: {parquet_path}")

    # 2. Feather (fastest, keeps geometry)
    df_to_save.to_feather(feather_path)
    print(f"Saved Feather: {feather_path}")

    # 3. CSV (simple, no geometry preserved)
    df_to_save.drop(columns='geometry').to_csv(csv_path, index=False)
    print(f"Saved CSV: {csv_path}")
else:
    print("csb_data not loaded, skipping save step.")

# 4. Create Monthly Data Structure (No Geometry)

In [ ]:
import pandas as pd
import numpy as np

if 'df_to_save' in locals():
    print("Starting monthly expansion...")
    
    # 1. Define new output directory for monthly data
    monthly_folder_name = f"csb_monthly_{START_YEAR}{END_YEAR}"
    monthly_output_dir = os.path.join(OUTPUT_BASE_DIR, monthly_folder_name)
    os.makedirs(monthly_output_dir, exist_ok=True)
    
    # 2. Prepare the Base Data (Drop Geometry & Static Spatial Cols)
    # We only need the ID and the Crop info. 
    # CNTY is kept to help merge with county-level weather/econ data later.
    cols_to_drop = ['geometry', 'INSIDE_X', 'INSIDE_Y', 'Shape_Length', 'Shape_Area', 'CSBYEARS', 'CSBACRES']
    monthly_base = df_to_save.drop(columns=cols_to_drop, errors='ignore')
    
    # 3. Melt: Convert "CDL2017, CDL2018..." columns into rows
    cdl_cols = [c for c in monthly_base.columns if c.startswith('CDL')]
    melted = monthly_base.melt(
        id_vars=['CSBID', 'CNTY', 'STATEFIPS'], # Identifiers to keep
        value_vars=cdl_cols, 
        var_name='Year_Raw', 
        value_name='Crop_Code'
    )
    
    # Clean up Year column
    melted['Year'] = melted['Year_Raw'].str.replace('CDL', '').astype(int)
    melted = melted.drop(columns=['Year_Raw'])
    
    print(f"Annual Long Format created: {len(melted)} rows")
    
    # 4. Expand: Replicate each row 12 times for Months 1-12
    # This is the most efficient way to do this in Pandas
    expanded = melted.loc[melted.index.repeat(12)].reset_index(drop=True)
    
    # Assign months (1, 2, ... 12, 1, 2, ... 12)
    expanded['Month'] = np.tile(np.arange(1, 13), len(melted))
    
    # 5. Save to Parquet
    # We use Parquet because this file will be large (approx 12x larger row count)
    monthly_file_name = f"ny_csb_monthly_{START_YEAR}{END_YEAR}.parquet"
    monthly_path = os.path.join(monthly_output_dir, monthly_file_name)
    
    print(f"Saving monthly structure ({len(expanded)} rows) to {monthly_path}...")
    expanded.to_parquet(monthly_path, index=False)
    
    print("Done! You can now merge weather data onto this file using [CSBID, Year, Month] or [CNTY, Year, Month].")

else:
    print("df_to_save not found. Run the previous cells first.")